# Política de Crédito

O objetivo deste notebook é transformar o modelo desenvolvido nas etapas anteriores em uma ferramenta de apoio à decisão de crédito.

Inicialmente, o modelo selecionado será utilizado para escorar as bases de desenvolvimento e validação, permitindo analisar o comportamento das probabilidades estimadas e construir uma política de crédito baseada em risco.

Posteriormente, a mesma pipeline será aplicada à base de submissão, gerando as probabilidades finais de inadimplência solicitadas no desafio.

Ao final do processo será gerado o arquivo:

`submissao_case.csv`

contendo: `id_cliente` e `probabilidade_inadimplencia`.

## 1. Imports e Configurações

In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path('..').resolve()))

import os
import pickle

import numpy as np
import pandas as pd

from src.feature_engineering import (
    feature_engineering
)
from src.policy import (
    build_rating_policy,
    apply_rating_policy,
    policy_report,
    compare_policy
)

TARGET = "target"
RANDOM_STATE = 42

## 2. Carregamento das Bases e Artefatos

### 2.1 Bases

Neste notebook serão utilizadas quatro bases:

- **Train:** Base completa de desenvolvimento após Feature Engineering e antes do balanceamento.
Objetivo: Construção da Política de Crédito

- **OOT:** Base temporal utilizada para validação da política.
Objetivo: Validação da estabilidade das faixas

- **Population Score:** Base sem target utilizada para geração das previsões finais.
Objetivo: Geração do arquivo submissao_case.csv


In [2]:
train_model = pd.read_parquet(
    "../data/processed/train_model.parquet"
)

oot_model = pd.read_parquet(
    "../data/processed/oot_model.parquet"
)

population_score = pd.read_parquet(
    "../data/processed/population_score.parquet"
)

### 2.2 Artefatos

E também serão carregados os artefatos produzidos nos notebooks anteriores:

- dicionario_imputacao.pkl
- dicionario_dominio.pkl
- woe_dictionary.pkl
- normalization_dictionary.pkl
- best_model.pkl
- best_model_features.pkl

In [3]:
with open(
    "../outputs/dicts/dicionario_imputacao.pkl",
    "rb"
) as f:
    dicionario_imputacao = pickle.load(f)

with open(
    "../outputs/dicts/dicionario_dominio.pkl",
    "rb"
) as f:
    dicionario_dominio = pickle.load(f)

with open(
    "../outputs/dicts/woe_dictionary.pkl",
    "rb"
) as f:
    woe_dictionary = pickle.load(f)

with open(
    "../outputs/dicts/normalization_dictionary.pkl",
    "rb"
) as f:
    normalization_dictionary = pickle.load(f)

with open(
    "../outputs/models/best_model.pkl",
    "rb"
) as f:
    best_model = pickle.load(f)

with open(
    "../outputs/dicts/best_model_features.pkl",
    "rb"
) as f:
    selected_features = pickle.load(f)

## 3. Escoragem

### 3.1 Base de Treino e OOT

O objetivo desta etapa é aplicar o modelo às bases históricas para compreender o comportamento dos scores e construir a política de crédito.

In [4]:
## Treino
train_scored = train_model.copy()

train_scored["probabilidade_inadimplencia"] = (
    best_model
    .predict_proba(
        train_scored[
            selected_features
        ]
    )[:,1]
)

In [5]:
## OOT
oot_scored = oot_model.copy()

oot_scored["probabilidade_inadimplencia"] = (
    best_model
    .predict_proba(
        oot_scored[
            selected_features
        ]
    )[:,1]
)

### 3.2 Base de Submissão

Primeiro aplicamos exatamente a mesma Feature Engineering utilizada durante o treinamento.

In [6]:
population_score_fe = (
    feature_engineering(
        population_score.copy(),
        dicionario_imputacao,
        dicionario_dominio,
        woe_dictionary,
        normalization_dictionary
    )
)

population_score_fe[
    "probabilidade_inadimplencia"
] = (
    best_model
    .predict_proba(
        population_score_fe[
            selected_features
        ]
    )[:,1]
)

## 4. Política de Crédito

A política será construída utilizando decis de risco.

A probabilidade de inadimplência será dividida em 10 grupos com quantidades semelhantes de clientes.

In [7]:
train_scored["decil"] = pd.qcut(
    train_scored[
        "probabilidade_inadimplencia"
    ],
    q=10,
    labels=False
) + 1

### 4.1 Performance por Decil

In [8]:
credit_policy = (
    train_scored
    .groupby("decil")
    .agg(
        clientes=(
            TARGET,
            "count"
        ),
        inadimplencia=(
            TARGET,
            "mean"
        ),
        score_min=(
            "probabilidade_inadimplencia",
            "min"
        ),
        score_max=(
            "probabilidade_inadimplencia",
            "max"
        )
    )
    .reset_index()
)

### 4.2 Construção dos Ratings

Transformar os decis em ratings.

Exemplo:

| Rating | Faixa       |
| ------ | ----------- |
| A      | Menor risco |
| B      |             |
| C      |             |
| D      |             |
| E      | Maior risco |


In [16]:
rating_bins, rating_labels = (
    build_rating_policy(
        train_scored
    )
)

train_scored = (
    apply_rating_policy(
        train_scored,
        rating_bins,
        rating_labels
    )
)

train_scored.head()

,id_cliente,valor_credito_submissao,valor_bem_submissao,valor_parcela_submissao,qtd_filhos,qtd_membros_familia,renda_anual,possui_carro,possui_imovel,nota_regiao_cliente,...,hora_solicitacao_submissao_woe,estado_civil_woe,tipo_renda_woe,tipo_organizacao_woe,tipo_moradia_woe,nivel_educacao_woe,probabilidade_inadimplencia,decil,rating,acao
0,310937,0.242385,0.203416,0.301683,0.666667,0.75,0.269663,1,1,0.5,...,0.690881,0.179734,0.941978,0.607036,0.772721,0.890073,0.735470,9,E,Reprovação
1,102358,0.990872,1.000000,1.000000,0.000000,0.25,0.303371,0,0,0.5,...,0.690881,0.650276,0.968614,0.607036,0.772721,1.000000,0.696217,9,E,Reprovação
2,292322,0.053564,0.052991,0.207587,0.666667,0.75,0.134831,0,0,0.0,...,0.690881,0.650276,0.941978,0.642046,0.772721,0.890073,0.438219,5,C,Análise Simplificada
3,333106,0.029368,0.066627,0.052881,0.000000,0.00,0.000000,0,1,0.5,...,0.690881,0.000000,0.892662,0.580620,0.772721,0.890073,0.590302,7,D,Análise Manual
4,186079,0.001584,0.002238,0.023275,0.333333,0.25,0.213483,0,0,1.0,...,0.690881,0.261775,0.941978,0.531509,0.709443,1.000000,0.250785,3,B,Aprovação Automática


### 4.3 Política Proposta

| Rating | Ação                               |
| ------ | ---------------------------------- |
| A      | Aprovação automática               |
| B      | Aprovação automática               |
| C      | Aprovação com análise simplificada |
| D      | Análise manual obrigatória         |
| E      | Reprovação                         |

In [10]:
rating_policy_train = (
    policy_report(
        train_scored,
        TARGET
    )
)

display(
    rating_policy_train
)

,rating,acao,clientes,bads,inadimplencia,score_min,score_max
0,A,Aprovação Automática,9906,0,0.000000,0.005647,0.243731
1,B,Aprovação Automática,9906,2,0.020190,0.243737,0.389439
2,C,Análise Simplificada,9905,22,0.222110,0.389455,0.524022
3,D,Análise Manual,9906,111,1.120533,0.524085,0.669942
4,E,Reprovação,9906,272,2.745811,0.669968,0.976276


### 4.4 Aplicação da Política na Base OOT

In [17]:
oot_scored = (
    apply_rating_policy(
        oot_scored,
        rating_bins,
        rating_labels
    )
)

oot_scored.head()

,id_cliente,valor_credito_submissao,valor_bem_submissao,valor_parcela_submissao,qtd_filhos,qtd_membros_familia,renda_anual,possui_carro,possui_imovel,nota_regiao_cliente,...,dia_semana_solicitacao_submissao_woe,hora_solicitacao_submissao_woe,estado_civil_woe,tipo_renda_woe,tipo_organizacao_woe,tipo_moradia_woe,nivel_educacao_woe,probabilidade_inadimplencia,rating,acao
0,100165,0.349824,0.348249,0.456578,0.000000,0.25,0.314607,1,1,0.5,...,0.091186,0.690881,0.650276,0.892662,0.580620,0.772721,0.890073,0.179804,A,Aprovação Automática
1,100567,0.072169,0.088200,0.134467,0.000000,0.25,0.123596,1,1,0.5,...,0.123463,0.690881,0.650276,0.941978,0.607036,0.772721,0.890073,0.824212,E,Reprovação
2,100591,0.323022,0.275832,0.315977,0.000000,0.25,0.325843,1,1,0.5,...,0.748463,0.690881,0.650276,0.968614,0.531509,0.772721,0.890073,0.386806,B,Aprovação Automática
3,100654,0.012984,0.024680,0.033570,0.666667,0.75,0.213483,0,1,0.5,...,0.305455,0.690881,0.650276,0.941978,0.527823,0.772721,1.000000,0.651859,D,Análise Manual
4,100675,0.995917,0.927583,0.539704,0.000000,0.25,0.382022,1,1,1.0,...,0.123463,0.690881,0.650276,0.892662,0.580620,0.772721,0.890073,0.203386,A,Aprovação Automática


In [ ]:
rating_policy_oot = (
    policy_report(
        oot_scored,
        TARGET
    )
)

display(
    rating_policy_oot
)

,rating,acao,clientes,bads,inadimplencia,score_min,score_max
0,A,Aprovação Automática,5509,2,0.036304,0.015322,0.243720
1,B,Aprovação Automática,4439,1,0.022528,0.243867,0.389366
2,C,Análise Simplificada,3672,3,0.081699,0.389458,0.524063
3,D,Análise Manual,3392,3,0.088443,0.524109,0.669921
4,E,Reprovação,2958,2,0.067613,0.669956,0.973935


### 4.5 Comparação Train x OOT

In [12]:
comparison_policy = (
    compare_policy(
        rating_policy_train,
        rating_policy_oot
    )
)

display(
    comparison_policy
)

,rating,inadimplencia_train,inadimplencia_oot
0,A,0.000000,0.036304
1,B,0.020190,0.022528
2,C,0.222110,0.081699
3,D,1.120533,0.088443
4,E,2.745811,0.067613


### 4.6 Aplicação na Base de Submissão

In [18]:
population_score_fe = (
    apply_rating_policy(
        population_score_fe,
        rating_bins,
        rating_labels
    )
)

population_score_fe.head()

,id_cliente,valor_credito_submissao,valor_bem_submissao,valor_parcela_submissao,qtd_filhos,qtd_membros_familia,renda_anual,possui_carro,possui_imovel,nota_regiao_cliente,...,dia_semana_solicitacao_submissao_woe,hora_solicitacao_submissao_woe,estado_civil_woe,tipo_renda_woe,tipo_organizacao_woe,tipo_moradia_woe,nivel_educacao_woe,probabilidade_inadimplencia,rating,acao
0,100023,0.397456,0.351870,0.223806,0.333333,0.25,0.101124,0,1,0.5,...,0.748463,0.690881,0.261775,1.000000,0.616471,0.772721,1.000000,0.632381,D,Análise Manual
1,100031,0.725131,0.551016,0.362838,0.000000,0.00,0.157303,0,1,1.0,...,0.748463,0.690881,0.000000,0.941978,0.642046,0.772721,0.890073,0.754059,E,Reprovação
2,100056,1.000000,1.000000,0.696804,0.000000,0.25,0.775281,1,1,0.5,...,0.000000,0.690881,0.650276,0.941978,1.000000,0.772721,0.890073,0.246435,B,Aprovação Automática
3,100067,0.021635,0.022374,0.045116,0.333333,0.50,0.280899,1,1,0.5,...,0.305455,0.690881,0.179734,0.941978,0.986825,0.772721,1.000000,0.197815,A,Aprovação Automática
4,100069,0.469663,0.402562,0.365600,0.333333,0.25,0.775281,1,1,0.5,...,0.748463,0.690881,1.000000,0.941978,0.704566,0.772721,0.890073,0.251311,B,Aprovação Automática


In [ ]:
display(
    population_score_fe[
        [
            "id_cliente",
            "probabilidade_inadimplencia",
            "rating",
            "acao"
        ]
    ].head()
)

,id_cliente,probabilidade_inadimplencia,rating,acao
0,100023,0.632381,D,Análise Manual
1,100031,0.754059,E,Reprovação
2,100056,0.246435,B,Aprovação Automática
3,100067,0.197815,A,Aprovação Automática
4,100069,0.251311,B,Aprovação Automática


## 5. Geração da Submissão

In [14]:
submission = (
    population_score_fe[
        [
            "id_cliente",
            "probabilidade_inadimplencia"
        ]
    ]
    .copy()
)

submission.head()

,id_cliente,probabilidade_inadimplencia
0,100023,0.632381
1,100031,0.754059
2,100056,0.246435
3,100067,0.197815
4,100069,0.251311


In [15]:
submission.to_csv(
    "../outputs/submissions/submissao_case.csv",
    index=False
)

## 6. Tabela Final da Política

In [19]:
politica_final = (
    rating_policy_train[
        [
            "rating",
            "acao",
            "score_min",
            "score_max",
            "inadimplencia"
        ]
    ]
    .copy()
)

politica_final = politica_final.rename(
    columns={
        "score_min": "prob_min",
        "score_max": "prob_max",
        "inadimplencia": "inadimplencia_observada"
    }
)

politica_final["inadimplencia_observada"] = (
    politica_final[
        "inadimplencia_observada"
    ].round(2)
)

display(
    politica_final
)

,rating,acao,prob_min,prob_max,inadimplencia_observada
0,A,Aprovação Automática,0.005647,0.243731,0.00
1,B,Aprovação Automática,0.243737,0.389439,0.02
2,C,Análise Simplificada,0.389455,0.524022,0.22
3,D,Análise Manual,0.524085,0.669942,1.12
4,E,Reprovação,0.669968,0.976276,2.75


## 7. Conclusão

Ao longo deste projeto foi desenvolvida uma solução completa de modelagem de risco de crédito, contemplando todas as etapas do ciclo analítico: definição da população, construção da variável alvo, análise exploratória dos dados, engenharia de atributos, seleção de variáveis, modelagem preditiva e construção de uma política de crédito baseada em risco.

Após a comparação dos modelos candidatos, o **LightGBM** foi selecionado como modelo final por apresentar o melhor desempenho nas métricas de validação, especialmente no indicador **KS**, demonstrando maior capacidade de separação entre clientes adimplentes e inadimplentes quando comparado aos demais algoritmos avaliados.

Com base nas probabilidades estimadas pelo modelo, foi construída uma política de crédito segmentada em cinco níveis de risco (Ratings A a E). Os cortes foram definidos a partir da distribuição dos scores observados na base de desenvolvimento, permitindo transformar as probabilidades de inadimplência em regras objetivas de negócio.

A política proposta apresentou comportamento monotônico crescente da inadimplência observada entre os ratings, indicando que o modelo foi capaz de ordenar adequadamente os clientes conforme seu nível de risco.

| Rating | Ação Recomendada     | Probabilidade Mínima | Probabilidade Máxima | Inadimplência Observada (%) |
| ------ | -------------------- | -------------------- | -------------------- | --------------------------- |
| A      | Aprovação Automática | 0.005647             | 0.243731             | 0.00                        |
| B      | Aprovação Automática | 0.243737             | 0.389439             | 0.02                        |
| C      | Análise Simplificada | 0.389455             | 0.524022             | 0.22                        |
| D      | Análise Manual       | 0.524085             | 0.669942             | 1.12                        |
| E      | Reprovação           | 0.669968             | 0.976276             | 2.75                        |

Observa-se que os ratings A e B apresentam níveis extremamente baixos de inadimplência, tornando-os candidatos naturais para aprovação automática. O rating C representa uma faixa intermediária de risco, recomendando uma análise simplificada. Já os ratings D e E concentram os maiores níveis de inadimplência observada, justificando análises mais rigorosas ou até mesmo a reprovação automática das propostas de crédito.

Por fim, a mesma pipeline utilizada durante o desenvolvimento foi aplicada à base de submissão, garantindo consistência entre treinamento e produção. O resultado final foi a geração do arquivo **submissao_case.csv**, contendo as probabilidades estimadas de inadimplência para todos os clientes avaliados.

Dessa forma, o projeto entrega não apenas um modelo preditivo, mas uma proposta completa de política de crédito orientada por dados, capaz de apoiar decisões de concessão de crédito de forma objetiva, escalável e alinhada às boas práticas de gestão de risco.
